In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [3]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
teams.sort()
print(teams)

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GBP', 'HOU', 'IND', 'JAC', 'KCC', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NEP', 'NOS', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SFO', 'TBB', 'TEN', 'WAS']


In [52]:
df_fd_temp = pd.read_csv("data/nfl/FanDuel-NFL-2021 ET-10 ET-10 ET-64897-players-list.csv")
df_fd = df_fd_temp.convert_dtypes()
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,64897-39280,RB,Derrick,Derrick Henry,Henry,24.125,4,10400,TEN@JAC,TEN,JAC,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
1,64897-55050,RB,Christian,Christian McCaffrey,McCaffrey,16.800001,3,10000,PHI@CAR,CAR,PHI,D,Hamstring,<NA>,<NA>,<NA>,RB/FLEX
2,64897-54140,RB,Dalvin,Dalvin Cook,Cook,12.866666,3,9000,DET@MIN,MIN,DET,Q,Ankle,<NA>,<NA>,<NA>,RB/FLEX
3,64897-42104,RB,Alvin,Alvin Kamara,Kamara,13.225,4,8600,NO@WAS,NO,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
4,64897-63484,QB,Kyler,Kyler Murray,Murray,27.955,4,8500,SF@ARI,ARI,SF,<NA>,<NA>,<NA>,<NA>,<NA>,QB


In [53]:
replace_dict = {"GBP":"GB", "KCC":"KC", "LVR":"LV", "NEP":"NE", "NOS":"NO", "SFO":"SF", "TBB":"TB"}
clean_teams = [replace_dict.get(item,item) for item in teams]
print(clean_teams)

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAC', 'KC', 'LAC', 'LAR', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [54]:
df_odds = dict(zip(clean_teams, odds))
#print(df_odds)
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

{'CHI': '15.50', 'IND': '18.50', 'CIN': '18.75', 'TEN': '19.50', 'JAC': '19.75', 'MIN': '20.25', 'HOU': '20.50', 'MIA': '20.50', 'BAL': '21.25', 'DAL': '21.25', 'PIT': '21.50', 'GB': '22.25', 'NO': '22.50', 'NYJ': '22.75', 'LV': '23.00', 'CAR': '23.50', 'BUF': '23.75', 'LAR': '24.00', 'DEN': '24.25', 'NYG': '24.50', 'ATL': '25.25', 'NE': '25.75', 'WAS': '26.50', 'DET': '26.75', 'LAC': '27.00', 'SEA': '27.00', 'SF': '27.00', 'ARI': '27.75', 'CLE': '28.75', 'TB': '29.50', 'KC': '29.75', 'PHI': '29.75'}


In [55]:
df_fd_def = df_fd.query("Position == 'D'")
df_fd = df_fd.query("Position != 'D'")
df_fd_def.head(3)

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
223,64897-12541,D,New England,New England Patriots,Patriots,7.25,4,5000,NE@HOU,NE,HOU,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
255,64897-12551,D,Tampa Bay,Tampa Bay Buccaneers,Buccaneers,7.5,4,4800,MIA@TB,TB,MIA,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
273,64897-12546,D,Arizona,Arizona Cardinals,Cardinals,10.75,4,4700,SF@ARI,ARI,SF,<NA>,<NA>,<NA>,<NA>,<NA>,DEF


In [56]:
df_fd["expts"] = df_fd["Team"].map(df_odds)
df_fd.expts = df_fd.expts.astype('float')
pts_bins = [1.2, 1.1, 1.0, .9,.8]
df_fd["rkpts"] = pd.cut(df_fd.expts, 5, labels=pts_bins)
df_fd.rkpts = df_fd.rkpts.astype('float')
df_fd["new_expts"] = df_fd.FPPG * df_fd.rkpts
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position,expts,rkpts,new_expts
0,64897-39280,RB,Derrick,Derrick Henry,Henry,24.125,4,10400,TEN@JAC,TEN,JAC,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX,19.50,1.1,26.5375
1,64897-55050,RB,Christian,Christian McCaffrey,McCaffrey,16.800001,3,10000,PHI@CAR,CAR,PHI,D,Hamstring,<NA>,<NA>,<NA>,RB/FLEX,23.50,1.0,16.800001
2,64897-54140,RB,Dalvin,Dalvin Cook,Cook,12.866666,3,9000,DET@MIN,MIN,DET,Q,Ankle,<NA>,<NA>,<NA>,RB/FLEX,20.25,1.1,14.153333
3,64897-42104,RB,Alvin,Alvin Kamara,Kamara,13.225,4,8600,NO@WAS,NO,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX,22.50,1.0,13.225
4,64897-63484,QB,Kyler,Kyler Murray,Murray,27.955,4,8500,SF@ARI,ARI,SF,<NA>,<NA>,<NA>,<NA>,<NA>,QB,27.75,0.8,22.364


In [57]:
df_fd.FPPG = df_fd.new_expts
df_fd = df_fd.drop(['expts', 'rkpts', 'new_expts'], axis=1)
df_fd = pd.concat([df_fd, df_fd_def])
#df_fd.query("Position == 'D'")
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
223,64897-12541,D,New England,New England Patriots,Patriots,7.25,4,5000,NE@HOU,NE,HOU,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
255,64897-12551,D,Tampa Bay,Tampa Bay Buccaneers,Buccaneers,7.5,4,4800,MIA@TB,TB,MIA,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
273,64897-12546,D,Arizona,Arizona Cardinals,Cardinals,10.75,4,4700,SF@ARI,ARI,SF,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
295,64897-12542,D,New Orleans,New Orleans Saints,Saints,10.75,4,4600,NO@WAS,NO,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
357,64897-12537,D,Las Vegas,Las Vegas Raiders,Raiders,4.25,4,4500,CHI@LV,LV,CHI,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
626,64897-12531,D,Denver,Denver Broncos,Broncos,9.75,4,4400,DEN@PIT,DEN,PIT,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
632,64897-12527,D,Chicago,Chicago Bears,Bears,8.5,4,4300,CHI@LV,CHI,LV,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
636,64897-12534,D,Tennessee,Tennessee Titans,Titans,2.0,4,4200,TEN@JAC,TEN,JAC,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
646,64897-12529,D,Cleveland,Cleveland Browns,Browns,8.0,4,4100,CLE@LAC,CLE,LAC,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
647,64897-12533,D,Green Bay,Green Bay Packers,Packers,4.0,4,4100,GB@CIN,GB,CIN,<NA>,<NA>,<NA>,<NA>,<NA>,DEF


In [58]:
df_fd.to_csv('data/nfl/FDprojections.csv')

In [1]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter, AfterEachExposureStrategy, TeamStack

optimizer = get_optimizer(Site.FANDUEL, Sport.FOOTBALL)
optimizer.load_players_from_csv("data/nfl/FDprojections.csv")
optimizer.add_stack(TeamStack(3, for_positions=['QB', 'WR', 'TE']))
#optimizer.add_stack(TeamStack(2, for_positions=['RB', 'D']))
for lineup in optimizer.optimize(40, max_exposure=0.3, exposure_strategy = AfterEachExposureStrategy):
    print(lineup)
optimizer.export('data/nfl/fd_result.csv')

 1. QB      Sam Darnold                   QB    CAR            PHI@CAR  24.44          7600.0$   
 2. RB      Derrick Henry                 RB    TEN            TEN@JAC  26.538         10400.0$  
 3. RB      James Robinson                RB    JAC            TEN@JAC  15.757         7400.0$   
 4. WR      Brandin Cooks                 WR    HOU            NE@HOU   15.785         6300.0$   
 5. WR      Brandon Zylstra               WR    CAR            PHI@CAR  9.6            4900.0$   
 6. WR      DJ Moore                      WR    CAR            PHI@CAR  18.675         7900.0$   
 7. TE      Dalton Schultz                TE    DAL            NYG@DAL  12.025         6200.0$   
 8. FLEX    John Ross III                 WR    NYG            NYG@DAL  13.68          5200.0$   
 9. DEF     Dallas Cowboys                D     DAL            NYG@DAL  8.75           4000.0$   

Fantasy Points 145.25
Salary 59900.00

 1. QB      Kirk Cousins                  QB    MIN            DET@MIN  22.974